In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from IPython.display import display

# **Cargar archivos**

In [ ]:
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")
gender_submission = pd.read_csv("/content/gender_submission.csv")
print("TRAIN")
print("Dimensiones:", train.shape)
display(train.head())

print("\nTEST")
print("Dimensiones:", test.shape)
display(test.head())

print("\nGENDER SUBMISSION")
print("Dimensiones:", gender_submission.shape)
display(gender_submission.head())

# Resumen de tipos, nulos y cardinalidad


In [ ]:
resumen = pd.DataFrame({
    'Tipo': train.dtypes.astype(str),
    'No nulos': train.notna().sum(),
    'Nulos': train.isna().sum(),
    '% Nulos': (train.isna().mean()*100).round(2),
    'Únicos': train.nunique(dropna=True)
})
display(resumen)

print('Filas duplicadas en train:', train.duplicated().sum())
print('Filas duplicadas en test :', test.duplicated().sum())

# Estadística descriptiva

In [ ]:
print('Resumen numérico de train.csv')
display(train.describe().T.round(2))

# Missing values de train y test


In [ ]:
missing_train = pd.DataFrame({
    'Missing': train.isna().sum(),
    'Porcentaje': (train.isna().mean()*100).round(2)
}).query('Missing > 0')

missing_test = pd.DataFrame({
    'Missing': test.isna().sum(),
    'Porcentaje': (test.isna().mean()*100).round(2)
}).query('Missing > 0')

print('Missing values - train.csv')
display(missing_train)
print('\nMissing values - test.csv')
display(missing_test)

# Gráfico de datos faltantes

In [ ]:
(train.isna().mean()*100).sort_values(ascending=False).plot(kind='bar', figsize=(10,4))
plt.title('Porcentaje de valores faltantes - train.csv')
plt.ylabel('Porcentaje (%)')
plt.xlabel('Variable')
plt.show()


# Distribución de la variable objetivo y patrones relevantes

In [ ]:
conteo = train['Survived'].value_counts().sort_index()
porcentaje = (train['Survived'].value_counts(normalize=True).sort_index()*100).round(2)
print('Conteo Survived:')
print(conteo)
print('\nPorcentaje Survived:')
print(porcentaje)

print('\nSupervivencia por sexo (%)')
print((train.groupby('Sex')['Survived'].mean()*100).round(2))

print('\nSupervivencia por clase (%)')
print((train.groupby('Pclass')['Survived'].mean()*100).round(2))

# Gráficos descriptivos

In [ ]:
(train.groupby('Sex')['Survived'].mean()*100).plot(kind='bar', figsize=(6,4))
plt.title('Tasa de supervivencia por sexo')
plt.ylabel('Supervivencia (%)')
plt.ylim(0,100)
plt.show()

(train.groupby('Pclass')['Survived'].mean()*100).plot(kind='bar', figsize=(6,4))
plt.title('Tasa de supervivencia por clase')
plt.ylabel('Supervivencia (%)')
plt.xlabel('Pclass')
plt.ylim(0,100)
plt.show()

def resumen_iqr(df, columna):
    s = df[columna].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    li = q1 - 1.5 * iqr
    ls = q3 + 1.5 * iqr
    outliers = s[(s < li) | (s > ls)]
    return pd.Series({
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'Límite inferior': li,
        'Límite superior': ls,
        'Cantidad atípicos': len(outliers),
        '% atípicos': round(len(outliers)/len(s)*100, 2)
    })

outliers_resumen = pd.DataFrame({
    'Age': resumen_iqr(train, 'Age'),
    'Fare': resumen_iqr(train, 'Fare')
}).T

display(outliers_resumen.round(2))

plt.figure(figsize=(8,4))
plt.boxplot([train['Age'].dropna(), train['Fare'].dropna()], tick_labels=['Age','Fare'])
plt.title('Boxplot de Age y Fare')
plt.show()

train_clean = train.copy()
test_clean = test.copy()

# Duplicados

In [ ]:
train_clean = train_clean.drop_duplicates().reset_index(drop=True)
test_clean = test_clean.drop_duplicates().reset_index(drop=True)


# Parámetros calculados desde train

In [ ]:
mediana_age = train['Age'].median()
moda_embarked = train['Embarked'].mode()[0]
mediana_fare = train['Fare'].median()

# Missing values

In [ ]:
for df in [train_clean, test_clean]:
    df['Age'] = df['Age'].fillna(mediana_age)
    df['Embarked'] = df['Embarked'].fillna(moda_embarked)
    df['Fare'] = df['Fare'].fillna(mediana_fare)

# Cabin: conservar sólo la información de presencia/ausencia

In [ ]:
for df in [train_clean, test_clean]:
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df.drop(columns=['Cabin'], inplace=True)

# Capping de Fare

In [ ]:
q1_fare = train['Fare'].quantile(0.25)
q3_fare = train['Fare'].quantile(0.75)
iqr_fare = q3_fare - q1_fare
limite_sup_fare = q3_fare + 1.5 * iqr_fare

train_clean['Fare'] = train_clean['Fare'].clip(upper=limite_sup_fare)
test_clean['Fare'] = test_clean['Fare'].clip(upper=limite_sup_fare)

print('Mediana Age:', mediana_age)
print('Moda Embarked:', moda_embarked)
print('Mediana Fare:', mediana_fare)
print('Límite superior Fare:', round(limite_sup_fare, 4))

# Verificación de limpieza

In [ ]:
print('Nulos restantes en train_clean:')
print(train_clean.isna().sum()[train_clean.isna().sum() > 0])
print('\nNulos restantes en test_clean:')
print(test_clean.isna().sum()[test_clean.isna().sum() > 0])

print('\nDimensiones finales:')
print('train_clean:', train_clean.shape)
print('test_clean :', test_clean.shape)
display(train_clean.head())


# Variables para un modelo base
# PassengerId, Name y Ticket se excluyen por identificación/alta cardinalidad.

In [ ]:
features = ['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked','HasCabin']
X = train_clean[features].copy()
y = train_clean['Survived'].copy()

# 70% train - 30% temporal

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# 15% validation - 15% test interno

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

resumen_split = pd.DataFrame({
    'Muestra': ['Train','Validation','Test interno'],
    'Registros': [len(X_train), len(X_val), len(X_test)],
    '% del total': [len(X_train)/len(X)*100, len(X_val)/len(X)*100, len(X_test)/len(X)*100],
    '% Survived=1': [y_train.mean()*100, y_val.mean()*100, y_test.mean()*100]
}).round(2)

display(resumen_split)


# Codificación simple para dejar las muestras listas para modelado

In [ ]:
X_train_model = pd.get_dummies(X_train, columns=['Sex','Embarked'], drop_first=True, dtype=int)
X_val_model = pd.get_dummies(X_val, columns=['Sex','Embarked'], drop_first=True, dtype=int)
X_test_model = pd.get_dummies(X_test, columns=['Sex','Embarked'], drop_first=True, dtype=int)

# Alinear las columnas

In [ ]:
X_val_model = X_val_model.reindex(columns=X_train_model.columns, fill_value=0)
X_test_model = X_test_model.reindex(columns=X_train_model.columns, fill_value=0)

print('Dimensiones para modelado:')
print('X_train:', X_train_model.shape)
print('X_val  :', X_val_model.shape)
print('X_test :', X_test_model.shape)

# En Colab se guardan en /content

In [ ]:
# Carpeta de salida en Google Colab
salida = "/content"

# Guardar bases limpias
train_clean.to_csv(f"{salida}/train_clean.csv", index=False)
test_clean.to_csv(f"{salida}/test_clean.csv", index=False)

# Crear archivos de entrenamiento, validación y test
train_split = X_train.copy()
train_split["Survived"] = y_train

val_split = X_val.copy()
val_split["Survived"] = y_val

test_split = X_test.copy()
test_split["Survived"] = y_test

# Guardar archivos
train_split.to_csv(f"{salida}/train_split.csv", index=False)
val_split.to_csv(f"{salida}/validation_split.csv", index=False)
test_split.to_csv(f"{salida}/test_interno_split.csv", index=False)

print("Archivos guardados correctamente en:", salida)

# Guardar mediante código su desarrollo en repositorio de Github.


In [189]:
import os
import subprocess
from getpass import getpass

# Datos de GitHub
usuario = "ignaciotoledo-arch"
repositorio = "analitica-datos-titanic"

# Token de GitHub
token = getpass("Pega tu Token de GitHub: ")

# Carpeta del proyecto
carpeta = "/content/proyecto_titanic"
os.makedirs(carpeta, exist_ok=True)

# Archivos que queremos subir
archivos = [
    "train_clean.csv",
    "test_clean.csv",
    "train_split.csv",
    "validation_split.csv",
    "test_interno_split.csv"
]

# Copiar archivos a la carpeta del proyecto
import shutil

for archivo in archivos:
    origen = f"/content/{archivo}"

    if os.path.exists(origen):
        shutil.copy(origen, carpeta)
        print("Preparado:", archivo)

# Entrar a la carpeta
os.chdir(carpeta)

# Configurar Git
subprocess.run(["git", "init"])
subprocess.run(["git", "branch", "-M", "main"])

subprocess.run([
    "git", "config",
    "user.name",
    usuario
])

subprocess.run([
    "git", "config",
    "user.email",
    f"{usuario}@users.noreply.github.com"
])

# Agregar archivos
subprocess.run(["git", "add", "."])

# Crear commit
subprocess.run(
    ["git", "commit", "-m", "Trabajo analítica de datos Titanic"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Dirección correcta de tu repositorio
url = f"https://github.com/{usuario}/{repositorio}.git"

subprocess.run(
    ["git", "remote", "remove", "origin"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

subprocess.run(["git", "remote", "add", "origin", url])

# Autenticación segura
askpass = "/tmp/git_askpass.sh"

with open(askpass, "w") as f:
    f.write("""#!/bin/sh
case "$1" in
    *Username*) echo "$GITHUB_USER" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")

os.chmod(askpass, 0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = askpass
env["GITHUB_USER"] = usuario
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"

# Subir a GitHub
resultado = subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    env=env,
    text=True,
    capture_output=True
)

if resultado.returncode == 0:
    print("\n======================================")
    print("ARCHIVOS SUBIDOS CORRECTAMENTE")
    print("======================================")
    print(f"https://github.com/{usuario}/{repositorio}")
else:
    print("\nERROR:")
    print(resultado.stderr)

Pega tu Token de GitHub: ··········
Preparado: train_clean.csv
Preparado: test_clean.csv
Preparado: train_split.csv
Preparado: validation_split.csv
Preparado: test_interno_split.csv

ERROR:
remote: Permission to ignaciotoledo-arch/analitica-datos-titanic.git denied to ignaciotoledo-arch.
fatal: unable to access 'https://github.com/ignaciotoledo-arch/analitica-datos-titanic.git/': The requested URL returned error: 403

